# Recommender Systems: Collaborative Filtering & Content-Based Filtering

Implementing both approaches from scratch in NumPy, following the math from
Andrew Ng's ML Specialization (Course 3, Week 2) — with derivations, manual
gradient computation, and honest observations on where each method holds up
and where it breaks.

## The problem

Given a sparse matrix of ratings (most users haven't rated most items), predict
the ratings a user *would* give to items they haven't seen yet. Two families
of solutions:

- **Collaborative filtering**: learn item features and user preferences
  simultaneously, using only the ratings — no side information needed.
- **Content-based filtering**: use known features of users and items (genre,
  demographics, etc.) to predict ratings via a learned similarity function.

We'll build both, compare them, and see where each falls apart.

## Notation

| Symbol | Meaning |
|---|---|
| $n_u$ | number of users |
| $n_m$ | number of movies |
| $n$ | number of features per movie/user |
| $r(i,j)$ | 1 if user $j$ has rated movie $i$, else 0 |
| $y^{(i,j)}$ | rating given by user $j$ to movie $i$ (defined only if $r(i,j)=1$) |
| $x^{(i)}$ | feature vector for movie $i$ (learned) |
| $w^{(j)}, b^{(j)}$ | parameter vector and bias for user $j$ (learned) |
| $\hat{y}^{(i,j)}$ | predicted rating: $w^{(j)} \cdot x^{(i)} + b^{(j)}$ |

The core difficulty: the ratings matrix $Y$ is **sparse** — most $(i,j)$ pairs
have $r(i,j) = 0$. Everything downstream (cost function, gradients) has to be
written to only look at observed entries.

## Collaborative Filtering — Cost Function

Predictions come from a bilinear form, same shape as linear regression, except
now *both* $x^{(i)}$ and $w^{(j)}$ are unknown:

$$\hat{y}^{(i,j)} = w^{(j)} \cdot x^{(i)} + b^{(j)}$$

Cost function, summed only over observed ratings ($r(i,j)=1$), with L2
regularization on both $w$ and $x$:

$$J(x, w, b) = \frac{1}{2} \sum_{(i,j):\, r(i,j)=1} \left( w^{(j)} \cdot x^{(i)} + b^{(j)} - y^{(i,j)} \right)^2
+ \frac{\lambda}{2} \sum_{j} \sum_{k} \left(w_k^{(j)}\right)^2
+ \frac{\lambda}{2} \sum_{i} \sum_{k} \left(x_k^{(i)}\right)^2$$

Note the symmetry: $x$ and $w$ are regularized identically, because neither is
"more true" than the other — both are being discovered jointly from the same
sparse signal.

## Collaborative Filtering — Gradients

Let $e^{(i,j)} = w^{(j)} \cdot x^{(i)} + b^{(j)} - y^{(i,j)}$ (the prediction error).

$$\frac{\partial J}{\partial x_k^{(i)}} = \sum_{j:\, r(i,j)=1} e^{(i,j)}\, w_k^{(j)} + \lambda x_k^{(i)}$$

$$\frac{\partial J}{\partial w_k^{(j)}} = \sum_{i:\, r(i,j)=1} e^{(i,j)}\, x_k^{(i)} + \lambda w_k^{(j)}$$

$$\frac{\partial J}{\partial b^{(j)}} = \sum_{i:\, r(i,j)=1} e^{(i,j)}$$

The pattern: the gradient w.r.t. $x^{(i)}$ looks exactly like a standard linear
regression gradient with $w^{(j)}$ playing the role of "features," and vice
versa. That symmetry is what makes joint gradient descent on both work.

## Why this works without knowing genres in advance

In a naive content-based approach you'd need someone to hand-label every
movie's genre mix before you could predict anything. Collaborative filtering
skips that entirely — it discovers a latent feature space purely from the
pattern of *who rated what how*. If two movies tend to get similar ratings
from the same users, gradient descent will push their learned $x^{(i)}$
vectors close together, even though nothing in the algorithm was ever told
what "genre" means.

The tradeoff, which we'll see later: a movie with **zero** ratings has
no gradient signal at all, so its features never move from initialization.
Collaborative filtering cannot recommend something nobody has rated yet.

## Implementation

We'll build a small synthetic dataset first — 10 movies, 6 users — small
enough to inspect by eye and small enough to gradient-check cheaply.

In [1]:
import numpy as np

np.random.seed(42)

In [2]:
n_movies = 10
n_users = 6
n_features = 3  # latent dimension we'll try to recover

# Ground-truth (unknown to the algorithm, used only to generate synthetic ratings)
true_X = np.random.randn(n_movies, n_features)
true_W = np.random.randn(n_users, n_features)
true_b = np.random.randn(n_users)

Y_full = true_X @ true_W.T + true_b  # (n_movies, n_users)

# Simulate sparsity: each user rates ~50% of movies
R = (np.random.rand(n_movies, n_users) < 0.5).astype(int)
Y = Y_full * R  # unrated entries are 0 but masked out by R downstream

print(f"Ratings matrix shape: {Y.shape}")
print(f"Fraction observed: {R.mean():.2f}")

Ratings matrix shape: (10, 6)
Fraction observed: 0.50


In [3]:
def cost_function(X, W, b, Y, R, lambda_):
    """
    X: (n_movies, n_features)
    W: (n_users, n_features)
    b: (n_users,)
    Y: (n_movies, n_users) ratings, R: (n_movies, n_users) mask
    """
    predictions = X @ W.T + b  # (n_movies, n_users)
    err = (predictions - Y) * R
    J = 0.5 * np.sum(err ** 2)
    J += (lambda_ / 2) * np.sum(W ** 2)
    J += (lambda_ / 2) * np.sum(X ** 2)
    return J

In [4]:
def compute_gradients(X, W, b, Y, R, lambda_):
    predictions = X @ W.T + b
    err = (predictions - Y) * R  # zero out unrated entries

    dX = err @ W + lambda_ * X          # (n_movies, n_features)
    dW = err.T @ X + lambda_ * W        # (n_users, n_features)
    db = np.sum(err, axis=0)            # (n_users,)

    return dX, dW, db

In [5]:
def train_collaborative_filter(Y, R, n_features, lambda_=1.0, alpha=0.01, n_iters=500):
    n_movies, n_users = Y.shape
    X = np.random.randn(n_movies, n_features) * 0.1
    W = np.random.randn(n_users, n_features) * 0.1
    b = np.zeros(n_users)

    history = []
    for it in range(n_iters):
        dX, dW, db = compute_gradients(X, W, b, Y, R, lambda_)
        X -= alpha * dX
        W -= alpha * dW
        b -= alpha * db

        if it % 50 == 0 or it == n_iters - 1:
            J = cost_function(X, W, b, Y, R, lambda_)
            history.append(J)
            print(f"Iter {it:4d} | Cost: {J:.4f}")

    return X, W, b, history

X_learned, W_learned, b_learned, history = train_collaborative_filter(Y, R, n_features)

Iter    0 | Cost: 50.6295
Iter   50 | Cost: 18.4386
Iter  100 | Cost: 10.1865
Iter  150 | Cost: 8.8285
Iter  200 | Cost: 8.6840
Iter  250 | Cost: 8.6167
Iter  300 | Cost: 8.5795
Iter  350 | Cost: 8.5590
Iter  400 | Cost: 8.5470
Iter  450 | Cost: 8.5389
Iter  499 | Cost: 8.5325


In [6]:
def numerical_gradient_check(X, W, b, Y, R, lambda_, epsilon=1e-4):
    """
    Compares analytical gradients against numerical ones on a handful of
    randomly sampled parameters. Should agree to ~1e-7 or better if the
    analytical gradient derivation is correct.
    """
    dX, dW, db = compute_gradients(X, W, b, Y, R, lambda_)

    # Check a few random entries in X
    print("Checking dX:")
    for _ in range(3):
        i, k = np.random.randint(X.shape[0]), np.random.randint(X.shape[1])
        X_plus, X_minus = X.copy(), X.copy()
        X_plus[i, k] += epsilon
        X_minus[i, k] -= epsilon
        numerical = (cost_function(X_plus, W, b, Y, R, lambda_) -
                     cost_function(X_minus, W, b, Y, R, lambda_)) / (2 * epsilon)
        analytical = dX[i, k]
        rel_err = abs(numerical - analytical) / (abs(numerical) + abs(analytical) + 1e-8)
        print(f"  [{i},{k}] numerical={numerical:.6f}  analytical={analytical:.6f}  rel_err={rel_err:.2e}")

    print("Checking dW:")
    for _ in range(3):
        j, k = np.random.randint(W.shape[0]), np.random.randint(W.shape[1])
        W_plus, W_minus = W.copy(), W.copy()
        W_plus[j, k] += epsilon
        W_minus[j, k] -= epsilon
        numerical = (cost_function(X, W_plus, b, Y, R, lambda_) -
                     cost_function(X, W_minus, b, Y, R, lambda_)) / (2 * epsilon)
        analytical = dW[j, k]
        rel_err = abs(numerical - analytical) / (abs(numerical) + abs(analytical) + 1e-8)
        print(f"  [{j},{k}] numerical={numerical:.6f}  analytical={analytical:.6f}  rel_err={rel_err:.2e}")

numerical_gradient_check(X_learned, W_learned, b_learned, Y, R, lambda_=1.0)

Checking dX:
  [7,1] numerical=-0.014304  analytical=-0.014304  rel_err=1.10e-10
  [7,0] numerical=0.003408  analytical=0.003408  rel_err=1.62e-12
  [7,2] numerical=0.016667  analytical=0.016667  rel_err=7.14e-11
Checking dW:
  [0,1] numerical=-0.000527  analytical=-0.000527  rel_err=4.91e-09
  [3,1] numerical=-0.024655  analytical=-0.024655  rel_err=6.69e-11
  [2,1] numerical=0.016937  analytical=0.016937  rel_err=8.93e-11


Rel error should sit around 1e-7 to 1e-9 — anything above ~1e-4 signals a bug
in the analytical gradient. This is the same sanity check used in the model
selection notebook; worth running any time a cost function is hand-derived
rather than autodiff'd.

In [7]:
predictions = X_learned @ W_learned.T + b_learned

# Compare predictions on RATED entries (what the model was trained on)
rated_mask = R == 1
rmse_rated = np.sqrt(np.mean((predictions[rated_mask] - Y[rated_mask]) ** 2))

# Compare predictions on UNRATED entries against the ground truth Y_full
# (this is only possible because we simulated the data — in real life you'd
# never have this ground truth for unrated pairs)
unrated_mask = R == 0
rmse_unrated = np.sqrt(np.mean((predictions[unrated_mask] - Y_full[unrated_mask]) ** 2))

print(f"RMSE on rated (seen) entries:   {rmse_rated:.4f}")
print(f"RMSE on unrated (held-out) entries: {rmse_unrated:.4f}")

RMSE on rated (seen) entries:   0.3941
RMSE on unrated (held-out) entries: 1.8432


In [8]:
# Cold-start check: what does the model predict for a movie with ZERO ratings?
R_test = R.copy()
R_test[0, :] = 0  # pretend movie 0 has never been rated by anyone

X_cold, W_cold, b_cold, _ = train_collaborative_filter(Y, R_test, n_features, n_iters=500)

print("Learned feature vector for the never-rated movie:")
print(X_cold[0])
print("\nCompare to a movie that WAS rated:")
print(X_cold[1])

Iter    0 | Cost: 50.3963
Iter   50 | Cost: 16.2339
Iter  100 | Cost: 8.9647
Iter  150 | Cost: 8.2218
Iter  200 | Cost: 7.9338
Iter  250 | Cost: 7.8672
Iter  300 | Cost: 7.8504
Iter  350 | Cost: 7.8406
Iter  400 | Cost: 7.8309
Iter  450 | Cost: 7.8200
Iter  499 | Cost: 7.8086
Learned feature vector for the never-rated movie:
[-0.00064744 -0.00071877 -0.00189194]

Compare to a movie that WAS rated:
[-0.32551445 -0.20359892 -0.0820762 ]


## Observations

- The never-rated movie's feature vector stayed essentially at its random
  initialization: [-0.00065, -0.00072, -0.00189], all three components
  within 0.002 of zero. Compare that to a movie that *was* rated, whose
  learned features are an order of magnitude larger: [-0.326, -0.204, -0.082].
  This is exactly what the gradient math predicts — every term in dX for
  row 0 is multiplied by the mask R, which is all zero for that movie, so
  the gradient is identically zero every iteration. The parameters aren't
  "poorly learned," they're literally untouched. 500 iterations of training
  ran and this movie experienced none of it.

- Note the near-zero values aren't exactly zero — they're small residuals
  from the initial `np.random.randn(...) * 0.1` scaling, slightly shrunk by
  the regularization term (which *does* still apply to every row of X,
  rated or not, since regularization doesn't check R). So regularization
  quietly pulls the untouched row towards zero even though no rating-error
  signal ever reaches it — a subtle detail that's easy to miss if you only
  look at the cost curve.

- Cost curve for the cold-start run (converging to 7.81) came out slightly
  *lower* than the original full-data run (converging to 8.53), even though
  we removed information. That's not a contradiction — cost is only summed
  over R==1 entries, and this run had one fewer movie's worth of rated pairs
  contributing error terms, so there's simply less to be wrong about. Lower
  cost here doesn't mean a "better" model; it means fewer terms in the sum.

- The structural takeaway: collaborative filtering has no mechanism to
  recommend something nobody has interacted with. It's not a tuning problem
  you can fix with more iterations or a different learning rate — the
  gradient is mathematically zero regardless. Section 6-7 (content-based
  filtering) exists specifically to route around this, using known features
  instead of ratings history, so a movie can get a real prediction on day one.

## Content-Based Filtering — Why

Collaborative filtering just demonstrated its own limit: a movie with zero
ratings gets a zero gradient, forever. No amount of training fixes that,
because the model has nothing to learn *from* for that row.

Content-based filtering sidesteps this by not relying on ratings history at
all for the item side. Instead, it uses known features — genre, release
year, cast, whatever's available — for both users and items, and learns a
function that maps those features into a shared space where dot-product
similarity predicts rating.

The tradeoff is real: you now need decent features up front, which
collaborative filtering never required. If your features are bad or missing,
this approach struggles in a completely different way than CF does.

## Two-Tower Architecture

Instead of learning a raw feature vector $x^{(i)}$ per movie directly (as CF
does), we now start from a known feature vector $x_u^{(j)}$ for user $j$ and
$x_m^{(i)}$ for movie $i$, and pass each through its own small neural network:

$$v_u^{(j)} = \text{NN}_{\text{user}}(x_u^{(j)}) \in \mathbb{R}^d$$
$$v_m^{(i)} = \text{NN}_{\text{movie}}(x_m^{(i)}) \in \mathbb{R}^d$$

Both towers output vectors of the *same* dimension $d$, even though their
input feature vectors can be completely different sizes and represent
different things — that's what lets the next step work.

Prediction is just a dot product of the two learned embeddings:

$$\hat{y}^{(i,j)} = v_u^{(j)} \cdot v_m^{(i)}$$

Cost function is the same squared-error-plus-regularization shape as CF,
just now the parameters being learned are the weights of both networks
rather than raw $x$ and $w$ vectors:

$$J = \frac{1}{2} \sum_{(i,j):\, r(i,j)=1} \left( v_u^{(j)} \cdot v_m^{(i)} - y^{(i,j)} \right)^2 + \text{(regularization on network weights)}$$

## Why build the MLP from scratch instead of using Keras

The rest of this repo is deliberately NumPy-only, forward and backward pass
both hand-written — that's the whole point of ML_From_Scratch. Two towers
sharing a loss through a dot product is a genuinely different backprop
shape than anything implemented so far here (linear/logistic regression,
anomaly detection): the gradient has to flow backward from a *single* scalar
loss, through the dot product, into *two separate* networks simultaneously.

Concretely, if $L = \frac{1}{2}(v_u \cdot v_m - y)^2$ and $e = v_u \cdot v_m - y$:

$$\frac{\partial L}{\partial v_u} = e \cdot v_m \qquad \frac{\partial L}{\partial v_m} = e \cdot v_u$$

Each of those gradients becomes the "upstream gradient" fed into standard
backprop through its respective tower's dense layers. That's the piece worth
implementing by hand — everything past that point (dense layer forward/backward)
is the same math as any other from-scratch MLP.

## Implementation plan

1. Synthetic user-feature and movie-feature data (different feature counts,
   on purpose, to prove the towers don't need matching input sizes)
2. A small dense-layer building block (forward + backward), reused for both towers
3. Two towers, each a stack of dense layers, both ending in the same output dim
4. Full forward pass: user tower -> movie tower -> dot product -> prediction
5. Full backward pass: loss -> dot product gradient -> both towers -> weight gradients
6. Training loop
7. Gradient check on this network specifically (different shape of gradient than CF, so worth re-verifying rather than assuming it transfers)

In [9]:
np.random.seed(42)

n_users_cb = 6
n_movies_cb = 10
user_feature_dim = 4   # e.g. age bucket, region, avg rating given, etc.
movie_feature_dim = 5  # e.g. year, genre one-hot-ish, avg rating received, etc.
embedding_dim = 3      # shared output dimension d for both towers

# Synthetic input features (NOT the same as the learned x/w from Section 3)
user_features = np.random.randn(n_users_cb, user_feature_dim)
movie_features = np.random.randn(n_movies_cb, movie_feature_dim)

# Reuse the same ground-truth-generation trick: define "true" embeddings,
# take dot products, then treat that as observed ratings with sparsity
true_user_emb = np.random.randn(n_users_cb, embedding_dim)
true_movie_emb = np.random.randn(n_movies_cb, embedding_dim)
Y_cb_full = true_movie_emb @ true_user_emb.T  # (n_movies, n_users)

R_cb = (np.random.rand(n_movies_cb, n_users_cb) < 0.6).astype(int)
Y_cb = Y_cb_full * R_cb

print(f"User features: {user_features.shape}, Movie features: {movie_features.shape}")

User features: (6, 4), Movie features: (10, 5)


In [10]:
def init_dense_layer(n_in, n_out):
    # small random init, standard for a from-scratch MLP
    W = np.random.randn(n_in, n_out) * np.sqrt(1.0 / n_in)
    b = np.zeros(n_out)
    return W, b

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

def dense_forward(A_prev, W, b):
    """A_prev: (batch, n_in) -> Z: (batch, n_out)"""
    Z = A_prev @ W + b
    return Z

def dense_backward(dZ, A_prev, W):
    """
    dZ: gradient w.r.t. this layer's pre-activation output, (batch, n_out)
    Returns: dW, db, dA_prev
    No batch-size normalization here — content_based_cost already normalizes
    once globally by n_ratings, so normalizing again per-layer would double-count.
    """
    dW = A_prev.T @ dZ
    db = np.sum(dZ, axis=0)
    dA_prev = dZ @ W.T
    return dW, db, dA_prev

In [11]:
def init_tower(input_dim, hidden_dim, output_dim):
    """Two-layer tower: input -> hidden (ReLU) -> output (linear)"""
    W1, b1 = init_dense_layer(input_dim, hidden_dim)
    W2, b2 = init_dense_layer(hidden_dim, output_dim)
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

def tower_forward(X, params):
    """
    X: (batch, input_dim)
    Returns final embedding plus cached intermediates needed for backward
    """
    Z1 = dense_forward(X, params["W1"], params["b1"])
    A1 = relu(Z1)
    Z2 = dense_forward(A1, params["W2"], params["b2"])  # linear output, no activation
    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2}
    return Z2, cache

def tower_backward(dZ2, params, cache):
    dW2, db2, dA1 = dense_backward(dZ2, cache["A1"], params["W2"])
    dZ1 = dA1 * relu_grad(cache["Z1"])
    dW1, db1, _ = dense_backward(dZ1, cache["X"], params["W1"])
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

In [12]:
hidden_dim = 8

user_tower = init_tower(user_feature_dim, hidden_dim, embedding_dim)
movie_tower = init_tower(movie_feature_dim, hidden_dim, embedding_dim)

def content_based_forward(user_features, movie_features, user_tower, movie_tower):
    v_u, user_cache = tower_forward(user_features, user_tower)   # (n_users, d)
    v_m, movie_cache = tower_forward(movie_features, movie_tower)  # (n_movies, d)
    predictions = v_m @ v_u.T  # (n_movies, n_users), matches Y_cb shape
    return predictions, v_u, v_m, user_cache, movie_cache

preds, v_u, v_m, user_cache, movie_cache = content_based_forward(
    user_features, movie_features, user_tower, movie_tower
)
print(f"Prediction matrix shape: {preds.shape}")  # should be (10, 6)

Prediction matrix shape: (10, 6)


In [13]:
def content_based_cost(predictions, Y, R, user_tower, movie_tower, lambda_):
    err = (predictions - Y) * R
    J = 0.5 * np.sum(err ** 2) / max(np.sum(R), 1)  # normalize by number of ratings

    reg = 0
    for tower in [user_tower, movie_tower]:
        reg += np.sum(tower["W1"] ** 2) + np.sum(tower["W2"] ** 2)
    J += (lambda_ / 2) * reg

    return J

J_init = content_based_cost(preds, Y_cb, R_cb, user_tower, movie_tower, lambda_=0.01)
print(f"Initial cost: {J_init:.4f}")

Initial cost: 1.9719


## Backward pass — the part that's actually new here

The dot product `predictions = v_m @ v_u.T` is where gradient has to split
and flow into two different networks. Given the error matrix
`err = (predictions - Y) * R` (masked, same as CF):

$$\frac{\partial J}{\partial v_m} = \text{err} \cdot v_u \qquad \frac{\partial J}{\partial v_u} = \text{err}^T \cdot v_m$$

These become the upstream gradients (`dZ2` equivalent) fed into each tower's
own `tower_backward`. Everything after that point is standard dense-layer
backprop, already written above — the only new math is these two lines.

In [14]:
def content_based_backward(predictions, Y, R, v_u, v_m, user_tower, movie_tower,
                             user_cache, movie_cache, lambda_):
    n_ratings = max(np.sum(R), 1)
    err = (predictions - Y) * R / n_ratings  # (n_movies, n_users), pre-scaled by normalization

    # split gradient at the dot product
    dv_m = err @ v_u          # (n_movies, d)
    dv_u = err.T @ v_m        # (n_users, d)

    user_grads = tower_backward(dv_u, user_tower, user_cache)
    movie_grads = tower_backward(dv_m, movie_tower, movie_cache)

    # add regularization gradient (only touches W1, W2, not biases)
    user_grads["dW1"] += lambda_ * user_tower["W1"]
    user_grads["dW2"] += lambda_ * user_tower["W2"]
    movie_grads["dW1"] += lambda_ * movie_tower["W1"]
    movie_grads["dW2"] += lambda_ * movie_tower["W2"]

    return user_grads, movie_grads

In [15]:
def update_tower(tower, grads, alpha):
    tower["W1"] -= alpha * grads["dW1"]
    tower["b1"] -= alpha * grads["db1"]
    tower["W2"] -= alpha * grads["dW2"]
    tower["b2"] -= alpha * grads["db2"]

def train_content_based(user_features, movie_features, Y, R, hidden_dim, embedding_dim,
                          lambda_=0.01, alpha=0.1, n_iters=1000):
    user_tower = init_tower(user_features.shape[1], hidden_dim, embedding_dim)
    movie_tower = init_tower(movie_features.shape[1], hidden_dim, embedding_dim)

    for it in range(n_iters):
        preds, v_u, v_m, user_cache, movie_cache = content_based_forward(
            user_features, movie_features, user_tower, movie_tower
        )
        user_grads, movie_grads = content_based_backward(
            preds, Y, R, v_u, v_m, user_tower, movie_tower, user_cache, movie_cache, lambda_
        )
        update_tower(user_tower, user_grads, alpha)
        update_tower(movie_tower, movie_grads, alpha)

        if it % 100 == 0 or it == n_iters - 1:
            J = content_based_cost(preds, Y, R, user_tower, movie_tower, lambda_)
            print(f"Iter {it:4d} | Cost: {J:.4f}")

    return user_tower, movie_tower

user_tower_trained, movie_tower_trained = train_content_based(
    user_features, movie_features, Y_cb, R_cb, hidden_dim, embedding_dim
)

Iter    0 | Cost: 1.6025
Iter  100 | Cost: 0.5777
Iter  200 | Cost: 0.2477
Iter  300 | Cost: 0.2394
Iter  400 | Cost: 0.2139
Iter  500 | Cost: 0.1824
Iter  600 | Cost: 0.1773
Iter  700 | Cost: 0.1668
Iter  800 | Cost: 0.1660
Iter  900 | Cost: 0.1594
Iter  999 | Cost: 0.1607


In [16]:
def gradient_check_content_based(user_features, movie_features, Y, R,
                                   user_tower, movie_tower, lambda_, epsilon=1e-4):
    preds, v_u, v_m, user_cache, movie_cache = content_based_forward(
        user_features, movie_features, user_tower, movie_tower
    )
    user_grads, movie_grads = content_based_backward(
        preds, Y, R, v_u, v_m, user_tower, movie_tower, user_cache, movie_cache, lambda_
    )

    def cost_with_perturbation(tower_name, param_name, i, j, delta):
        towers = {"user": user_tower, "movie": movie_tower}
        original = towers[tower_name][param_name][i, j]
        towers[tower_name][param_name][i, j] = original + delta
        p, _, _, _, _ = content_based_forward(user_features, movie_features, user_tower, movie_tower)
        J = content_based_cost(p, Y, R, user_tower, movie_tower, lambda_)
        towers[tower_name][param_name][i, j] = original  # restore
        return J

    print("Checking movie_tower W1:")
    for _ in range(3):
        i, j = np.random.randint(movie_tower["W1"].shape[0]), np.random.randint(movie_tower["W1"].shape[1])
        J_plus = cost_with_perturbation("movie", "W1", i, j, epsilon)
        J_minus = cost_with_perturbation("movie", "W1", i, j, -epsilon)
        numerical = (J_plus - J_minus) / (2 * epsilon)
        analytical = movie_grads["dW1"][i, j]
        rel_err = abs(numerical - analytical) / (abs(numerical) + abs(analytical) + 1e-8)
        print(f"  [{i},{j}] numerical={numerical:.6f}  analytical={analytical:.6f}  rel_err={rel_err:.2e}")

gradient_check_content_based(user_features, movie_features, Y_cb, R_cb,
                               user_tower_trained, movie_tower_trained, lambda_=0.01)

Checking movie_tower W1:
  [0,5] numerical=0.015476  analytical=0.015476  rel_err=1.80e-12
  [0,2] numerical=0.016601  analytical=0.016601  rel_err=4.63e-12
  [4,4] numerical=0.007593  analytical=0.007593  rel_err=2.83e-12


In [17]:
final_preds, v_u_final, v_m_final, _, _ = content_based_forward(
    user_features, movie_features, user_tower_trained, movie_tower_trained
)

rated_mask_cb = R_cb == 1
rmse_rated_cb = np.sqrt(np.mean((final_preds[rated_mask_cb] - Y_cb[rated_mask_cb]) ** 2))

unrated_mask_cb = R_cb == 0
rmse_unrated_cb = np.sqrt(np.mean((final_preds[unrated_mask_cb] - Y_cb_full[unrated_mask_cb]) ** 2))

print(f"RMSE on rated (seen) entries:       {rmse_rated_cb:.4f}")
print(f"RMSE on unrated (held-out) entries: {rmse_unrated_cb:.4f}")

RMSE on rated (seen) entries:       0.2742
RMSE on unrated (held-out) entries: 2.4219


In [18]:
# The real test: a movie that has NEVER been rated by anyone, but DOES have
# known features (genre, year, etc.) — this is the case CF structurally cannot handle.

new_movie_features = np.random.randn(1, movie_feature_dim)  # a "new" movie, features only
v_m_new, _ = tower_forward(new_movie_features, movie_tower_trained)

# Predict its rating for every existing user
new_movie_preds = v_m_new @ v_u_final.T  # (1, n_users)
print("Predicted ratings for a brand-new, never-rated movie:")
print(new_movie_preds)

print("\nCompare to predictions for an existing, previously-rated movie:")
print(final_preds[0])

Predicted ratings for a brand-new, never-rated movie:
[[ 1.02346329  1.51825825 -0.77968436  0.75465813 -0.72081437 -0.43318668]]

Compare to predictions for an existing, previously-rated movie:
[ 1.17330919  1.29954561 -0.63533084  0.6852309  -0.60443007 -0.36681248]


## Observations

- The never-rated movie DID get a real, non-degenerate prediction:
  [1.023, 1.518, -0.780, 0.755, -0.721, -0.433]. Compare to Section 4, where
  the CF cold-start movie's feature vector stayed at [-0.00065, -0.00072,
  -0.00189] — essentially untouched. Here the new movie's predictions are
  actually close in scale and sign pattern to an existing rated movie's
  predictions ([1.173, 1.300, -0.635, 0.685, -0.604, -0.367] — same rough
  shape: two positives, then alternating signs at similar magnitudes). This
  confirms the structural claim: v_m depends only on features (x_m), which
  the new movie has, not on rating history, which it doesn't. The fix works
  as designed.

- The RMSE numbers tell a different, more honest story than the single
  cold-start example above. RMSE on rated entries: 0.2742. RMSE on unrated
  entries: 2.4219 — almost 9x worse. That's a real generalization gap, not
  noise. It means: for the *specific* new movie tested above, the network
  happened to produce a sensible-looking prediction, but averaged across all
  unrated (movie, user) pairs, predictions are systematically far worse than
  on data the model actually trained on. One good-looking example doesn't
  contradict a bad aggregate RMSE — it just means results vary a lot
  pair-to-pair, and this pair happened to land well.

- Why the gap is probably real overfitting, not a bug: only 10 movies and
  6 users went into training two separate MLPs (each with a hidden layer of
  8 units) — that's a lot of parameters relative to how little data is
  actually observed after masking by R_cb. The gradient check confirmed the
  math is correct, so this is a capacity/data problem, not an implementation
  one. More data, a smaller hidden_dim, or stronger lambda_ would likely
  shrink this gap — worth trying if this were a real dataset rather than a
  from-scratch demo.

- The tradeoff from before still stands, now with a number attached to it:
  content-based filtering solves CF's zero-rating problem, but at a cost —
  this synthetic experiment's held-out RMSE (2.42) is noticeably worse than
  CF's held-out performance in Section 4. Small comparison worth remembering
  for Section 8: neither method's cold-start fix comes free, and content-
  based's fix here came with a real generalization penalty in this run.

## Collaborative Filtering vs Content-Based Filtering — When to Reach for Each

| | Collaborative Filtering | Content-Based Filtering |
|---|---|---|
| Input needed | Only ratings (no metadata required) | Feature vectors for users AND items |
| Cold-start (new item, zero ratings) | Fails structurally — zero gradient, features frozen at init (Section 4) | Works — features flow through the tower regardless of rating history (Section 7) |
| Cold-start (item has features but no metadata at all) | N/A — doesn't use features to begin with | Fails — no input, no embedding |
| Held-out RMSE in this notebook | Not directly measured the same way (different cost normalization) | 0.27 (rated) vs 2.42 (unrated) — real generalization gap |
| Discovers latent structure automatically | Yes — that's the whole appeal, no manual feature engineering | No — quality is capped by how good the input features are |
| Parameters learned | Raw x, w vectors — number scales with (movies + users) × features | Full MLP weights — much larger parameter count for same data size |

The practical takeaway from actually building both: CF is simpler, needs less
input, and is more data-efficient per parameter — but it's completely blind
to anything with zero ratings. Content-based fixes that specific blindness,
at the cost of needing decent metadata and, in this run at least, generalizing
noticeably worse on held-out pairs given the same tiny synthetic dataset.

**Hybrid approaches** (not implemented here) typically use CF as the default
and fall back to content-based specifically for new items/users — getting
CF's efficiency where there's rating history, and content-based's cold-start
coverage where there isn't.

## Final Observations

- The most surprising thing wasn't a result — it was the bug in Section 6.
  The gradient check initially failed with rel_err around 0.5-1.0 (nowhere
  close to a rounding issue), traced back to `dense_backward` dividing by
  batch size `m` on top of `content_based_cost` already normalizing by
  `n_ratings`. Two independent normalizations stacking silently. The model
  still "trained" with the bug in place — cost went down, numbers looked
  plausible — which is the uncomfortable part. Nothing about the training
  curve alone would have caught this; only the numerical gradient check did.
  Reinforces why that check belongs in every from-scratch notebook, not just
  the ones where something feels off.

- The RMSE gap in Section 7 (0.27 rated vs 2.42 unrated) was a good reminder
  that a single good-looking example — the new movie's prediction lining up
  nicely with an existing movie's — isn't evidence of general performance.
  It's easy to eyeball one output, see it looks sane, and move on. The
  aggregate metric told a less flattering story that the single example
  would have hidden.

- If continuing this: the first thing worth trying is shrinking hidden_dim
  or increasing lambda_ for the content-based towers, since the parameter
  count relative to 10 movies × 6 users is almost certainly why the held-out
  RMSE is as high as it is. Second would be actually implementing the hybrid
  fallback described in Section 8 — even a naive version (use CF prediction
  if the movie has >= 1 rating, else fall back to content-based) would be a
  natural next notebook.

- The cold-start parallel to intrusion detection is real, not forced: an IDS
  trained purely on observed traffic patterns (the CF-style approach) has
  the same structural blind spot as CF had for unrated movies — a genuinely
  novel attack pattern with zero prior examples gets no gradient signal,
  same as the never-rated movie in Section 4. Signature- or feature-based
  detection (the content-based-style approach) can flag something novel
  based on known suspicious characteristics even with zero prior instances
  of that exact attack — but it inherits content-based filtering's dependency
  on those features actually being informative. The zero-day problem in
  security and the cold-start problem in recommenders are the same shape of
  problem: no history, only structure.